# GPT-SoVITS Improve — Fast Full Install (Kaggle)

이 노트북은 **Kaggle에서 가능한 한 셀을 수정하지 않고 Run All** 하는 것을 목표로 합니다.

- Kaggle Dataset만 Notebook에 연결합니다.
- `datas/` 탐색, 선택적 `training.list` 처리, Preflight, WebUI 기본 경로 주입은 Git 저장소 코드가 담당합니다.
- `.list`가 없으면 WebUI가 긴 음성 자동 분할 → Faster Whisper 전사 → `.list` 생성을 수행합니다.
- repo·캐시·전처리 중간물은 `/kaggle/tmp`, 최종 모델만 `/kaggle/working`에 저장합니다.

Kaggle Notebook 설정에서 **GPU**와 **Internet**을 켠 뒤 위에서부터 실행하세요.


## 0. Dataset 연결만 하면 됩니다

현재 기본 우선순위는 `pymood1/w-voice-file`입니다. 저장소가 다음 경로들을 자동 탐색합니다.

```text
/kaggle/input/datasets/pymood1/w-voice-file/datas/
/kaggle/input/w-voice-file/datas/
```

둘 다 없으면 `/kaggle/input` 아래에서 음성이 들어 있는 `datas/`를 자동 탐색합니다.
`training.list`는 선택 사항입니다. **이 노트북에서 경로 코드를 수정할 필요가 없습니다.**


### 데이터 준비는 Git 코드가 담당

이전 버전의 긴 Dataset 탐색/압축해제/`training.list` 변환 셀은 제거했습니다.
`tools/kaggle_dataset.py`가 Dataset을 준비하고, `tools/kaggle_launch_webui.py`가 Preflight와 WebUI 실행까지 연결합니다.


In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys
import time


REPO_URL = "https://github.com/pymood1/GPT-SoVITS-improve.git"
REPO_DIR = Path("/kaggle/tmp/GPT-SoVITS")


print("[1/2] GPT-SoVITS-improve 최신 main 준비", flush=True)

start = time.perf_counter()

if (REPO_DIR / ".git").exists():
    print("[git] 기존 repo 유지 + 최신 main 동기화", flush=True)

    subprocess.run(
        ["git", "fetch", "--depth", "1", "origin", "main"],
        cwd=REPO_DIR,
        check=True,
    )

    subprocess.run(
        ["git", "reset", "--hard", "origin/main"],
        cwd=REPO_DIR,
        check=True,
    )

else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)

    print("[git] 새 repo shallow clone", flush=True)

    clone_cmd = [
        "git",
        "clone",
        "--depth",
        "1",
        "--single-branch",
        "--branch",
        "main",
        REPO_URL,
        str(REPO_DIR),
    ]

    subprocess.run(clone_cmd, check=True)


elapsed = time.perf_counter() - start
print(f"[1/2] repo 준비 완료: {elapsed:.1f}s", flush=True)


print("[2/2] 전체 의존성 + pretrained 설치 시작", flush=True)

setup_script = REPO_DIR / "tools" / "colab_fast_setup.py"

# upstream Colab 설치 로직은 그대로 사용하고,
# Colab 전용 작업 경로만 Kaggle working 경로로 최소 치환합니다.
setup_text = setup_script.read_text(encoding="utf-8")
setup_text = setup_text.replace("/content/", "/kaggle/tmp/")
setup_script.write_text(setup_text, encoding="utf-8")


print("[check] setup script 문법 검사", flush=True)

subprocess.run(
    [sys.executable, "-m", "py_compile", str(setup_script)],
    check=True,
)


print("[run] setup script 실행 — 최근 로그만 표시", flush=True)

from collections import deque
from IPython.display import clear_output

setup_log = Path("/kaggle/tmp/gpt_sovits_setup.log")
tail = deque(maxlen=24)
pending = ""
last_render = 0.0


def render_tail(final=False):
    global last_render
    now = time.monotonic()
    if not final and now - last_render < 0.8:
        return
    last_render = now
    clear_output(wait=True)
    print("[1/2] repo 준비 완료")
    print("[2/2] 전체 의존성 + pretrained 설치")
    print(f"[log] 전체 로그: {setup_log}")
    print("-" * 72)
    if tail:
        print("\n".join(tail))
    else:
        print("설치 프로세스 시작 중...")


with setup_log.open("w", encoding="utf-8") as log_writer:
    process = subprocess.Popen(
        [sys.executable, "-u", str(setup_script)],
        stdout=log_writer,
        stderr=subprocess.STDOUT,
        text=True,
    )

    with setup_log.open(
        "r",
        encoding="utf-8",
        errors="replace",
    ) as log_reader:
        while process.poll() is None:
            chunk = log_reader.read()
            if chunk:
                pending += chunk.replace("\r", "\n")
                parts = pending.split("\n")
                pending = parts.pop()
                for item in parts:
                    item = item.strip()
                    if item:
                        tail.append(item)
                render_tail()
            time.sleep(0.25)

        chunk = log_reader.read()
        if chunk:
            pending += chunk.replace("\r", "\n")

if pending:
    for item in pending.split("\n"):
        item = item.strip()
        if item:
            tail.append(item)

return_code = process.wait()
render_tail(final=True)

if return_code != 0:
    error_tail = setup_log.read_text(
        encoding="utf-8",
        errors="replace",
    ).replace("\r", "\n").splitlines()[-80:]
    clear_output(wait=True)
    print("\n".join(error_tail))
    raise RuntimeError(
        f"setup script failed with exit code {return_code}. "
        f"전체 로그: {setup_log}"
    )

print("\n[OK] 설치 완료", flush=True)


# Kaggle working disk 절약: 설치에만 필요한 다운로드/패키지 캐시 제거
print("[cleanup] Kaggle 설치 캐시 정리", flush=True)
cache_dirs = [
    Path("/kaggle/tmp/gpt_sovits_downloads"),
    Path("/kaggle/tmp/huggingface/hub"),
    Path("/root/.cache/pip"),
    Path("/root/.cache/uv"),
]
for cache_dir in cache_dirs:
    if cache_dir.exists():
        try:
            shutil.rmtree(cache_dir)
            print("  removed:", cache_dir)
        except OSError as exc:
            print("  warn:", cache_dir, exc)

for disk_path in ("/kaggle/tmp", "/kaggle/working"):
    usage = shutil.disk_usage(disk_path)
    print(
        f"[disk] {disk_path}: "
        f"{usage.free / 1024**3:.2f} GiB free / "
        f"{usage.total / 1024**3:.2f} GiB total"
    )


## 2. Preflight

별도 셀에서 데이터 경로를 전달할 필요가 없습니다. **WebUI launcher가 시작 전에 Preflight를 자동 실행**합니다.
GPU, pretrained, 핵심 Python 문법, synthetic self-test를 확인한 뒤 통과해야 WebUI를 시작합니다.


### 별도 Preflight 셀 불필요

`tools/kaggle_launch_webui.py` 내부에서 자동 실행됩니다.


## 3. Kaggle에서 실제 학습할 순서

WebUI가 뜨면 첫 실험은 아래 순서로 진행합니다.

```text
A0  pretrained S1 (학습 없음, 비교 기준)
A1  현재 Full FT baseline
B1  V + Attention Out LoRA / rank 8 / alpha 16
B2  QKV + Attention Out LoRA / rank 8 / alpha 16

각 S1 후보가 만들어질 때마다
→ S1 비교/진단
→ 해당 S1에 맞춰 Emotion Controller 새로 학습
→ OOF/청감 비교
```

**S2/SoVITS는 이 비교에서 고정**합니다. 서로 다른 S1 checkpoint끼리 Controller를 재사용하지 않습니다.

첫 LoRA 후보는 `V + Attention Out LoRA`, `rank=8`, `alpha=16`입니다. 실험 결과가 나오기 전까지 WebUI 기본값은 기존 Full FT를 유지합니다.


## V5dev 전체 WebUI — 한국어

이 셀은 저장소의 Kaggle launcher만 실행합니다.

launcher가 자동으로:

1. Kaggle Dataset의 `datas/`를 찾고
2. 기존 `training.list`가 있으면 정규화하고, 없으면 자동 생성 모드로 두고
3. Preflight를 실행하고
4. WebUI에 실제 데이터 경로를 환경변수로 전달하고
5. 한국어 V5dev WebUI를 시작합니다.

따라서 Notebook 셀에서 `webui.py`를 문자열 치환하지 않습니다.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_DIR = Path("/kaggle/tmp/GPT-SoVITS")
LAUNCHER = REPO_DIR / "tools" / "kaggle_launch_webui.py"

subprocess.run(
    [
        sys.executable,
        "-u",
        str(LAUNCHER),
    ],
    cwd=REPO_DIR,
    check=True,
)
